In [2]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\.01_projects\Tenderalpha_world_contract\unified-government-contract-awards")

files = [
    f for f in folder.glob("*.parquet")
    if not f.name.startswith("._")
]

print(files)

df = pd.read_parquet(files[0])

[WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-government-contract-awards_0_0_0.snappy.parquet'), WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-government-contract-awards_0_0_1.snappy.parquet'), WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-government-contract-awards_0_0_2.snappy.parquet'), WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-government-contract-awards_0_0_3.snappy.parquet'), WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-government-contract-awards_0_1_0.snappy.parquet'), WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-government-contract-awards_0_1_1.snappy.parquet'), WindowsPath('D:/.01_projects/Tenderalpha_world_contract/unified-government-contract-awards/unified-

## 合同国家与时间范围分析

下面根据 `TenderAlpha_UGCA_Data_Dictionary.md` 中的字段说明，分析当前 notebook 打开的 parquet 文件：合同履行国家、采购方国家、中标方国家、合同数量和时间范围。

In [4]:
import pandas as pd
from IPython.display import display

# 字段含义来自 TenderAlpha_UGCA_Data_Dictionary.md
analysis_cols = [
    "ORIGIN",                                  # 数据来源/采购区域
    "TENDER_BIZPORTAL_ID",                    # 合同/标的 ID
    "TRANSACTION_BIZPORTAL_ID",               # 合同授予交易 ID
    "CONTRACTING_ENTITY_COUNTRY",             # 采购方国家 ISO2
    "TENDER_FUNDING_ORIGIN_COUNTRY",          # 资金来源国家 ISO2
    "TENDER_COUNTRY",                         # 合同履行国家 ISO2
    "DIRECT_AWARDEE_COUNTRY",                 # 直接中标方国家 ISO2
    "AWARDEE_PARENT_COUNTRY",                 # 中标方母公司国家 ISO2
    "TENDER_DATE_OF_AWARD",                   # 授标日期
    "TENDER_DATE_OF_DISPATCH",                # 信息发布日期
    "TENDER_MIN_DELIVERY_DATE",               # 数据最早交付日期
    "TENDER_CONTRACT_START_DATE",             # 合同开始日期
    "TENDER_CONTRACT_END_DATE",               # 合同结束日期
]

# notebook 前面已经定义了 files 和 df；如果当前内核没有这些变量，则重新定位第一个真实 parquet 文件。
try:
    current_file = files[0]
except NameError:
    from pathlib import Path
    folder = Path(r"D:\.01_projects\Tenderalpha_world_contract\unified-government-contract-awards")
    files = sorted(f for f in folder.glob("*.parquet") if not f.name.startswith("._"))
    current_file = files[0]

try:
    ugca = df[analysis_cols].copy()
except NameError:
    ugca = pd.read_parquet(current_file, columns=analysis_cols)

print(f"当前分析文件: {current_file.name}")
print(f"记录行数: {len(ugca):,}")
print(f"唯一合同/标的 ID 数: {ugca['TENDER_BIZPORTAL_ID'].nunique(dropna=True):,}")
print(f"唯一交易 ID 数: {ugca['TRANSACTION_BIZPORTAL_ID'].nunique(dropna=True):,}")

当前分析文件: unified-government-contract-awards_0_0_0.snappy.parquet
记录行数: 1,272,627
唯一合同/标的 ID 数: 1,254,475
唯一交易 ID 数: 1,063,336


In [7]:
country_fields = {
    "CONTRACTING_ENTITY_COUNTRY": "采购方国家",
    "TENDER_FUNDING_ORIGIN_COUNTRY": "资金来源国家",
    "TENDER_COUNTRY": "合同履行国家",
    "DIRECT_AWARDEE_COUNTRY": "直接中标方国家",
    "AWARDEE_PARENT_COUNTRY": "中标方母公司国家",
}

country_overview = pd.DataFrame([
    {
        "字段": col,
        "含义": desc,
        "非空国家/地区数": ugca[col].nunique(dropna=True),
        "缺失记录数": ugca[col].isna().sum(),
        "缺失比例": ugca[col].isna().mean(),
    }
    for col, desc in country_fields.items()
])

display(country_overview.style.format({"缺失记录数": "{:,}", "缺失比例": "{:.2%}"}))

origin_counts = ugca["ORIGIN"].value_counts(dropna=False).rename_axis("ORIGIN").reset_index(name="合同记录数")
display(origin_counts.style.format({"合同记录数": "{:,}"}))

tender_country_counts = (
    ugca["TENDER_COUNTRY"]
    .value_counts(dropna=False)
    .rename_axis("TENDER_COUNTRY")
    .reset_index(name="合同记录数")
)
display(tender_country_counts.head(30).style.format({"合同记录数": "{:,}"}))

,字段,含义,非空国家/地区数,缺失记录数,缺失比例
0,CONTRACTING_ENTITY_COUNTRY,采购方国家,66,"3,000",0.24%
1,TENDER_FUNDING_ORIGIN_COUNTRY,资金来源国家,66,"2,583",0.20%
2,TENDER_COUNTRY,合同履行国家,213,"2,771",0.22%
3,DIRECT_AWARDEE_COUNTRY,直接中标方国家,176,"2,300",0.18%
4,AWARDEE_PARENT_COUNTRY,中标方母公司国家,87,"1,028,512",80.82%


,ORIGIN,合同记录数
0,US Procurement,"902,696"
1,South Korea Procurement,"152,935"
2,Russian Federation Procurement,"74,419"
3,Ukraine Procurement,"43,359"
4,EEA Procurement,"39,671"
5,Colombia Procurement,"14,024"
6,Chile Procurement,"9,645"
7,India Procurement,"8,476"
8,Australia Procurement,"7,903"
9,China Procurement,"6,056"


,TENDER_COUNTRY,合同记录数
0,US,"860,297"
1,KR,"154,483"
2,RU,"74,555"
3,UA,"43,456"
4,CO,"14,561"
5,RO,"10,018"
6,CL,"9,696"
7,IN,"8,704"
8,AU,"8,064"
9,DE,"7,979"


In [8]:
date_fields = {
    "TENDER_DATE_OF_AWARD": "授标日期",
    "TENDER_DATE_OF_DISPATCH": "信息发布日期",
    "TENDER_MIN_DELIVERY_DATE": "数据最早交付日期",
    "TENDER_CONTRACT_START_DATE": "合同开始日期",
    "TENDER_CONTRACT_END_DATE": "合同结束日期",
}

date_summary_rows = []
clean_start = pd.Timestamp("2009-01-01")
clean_end = pd.Timestamp("2026-12-31")

for col, desc in date_fields.items():
    s = pd.to_datetime(ugca[col], errors="coerce")
    clean = s[s.between(clean_start, clean_end)]
    date_summary_rows.append({
        "字段": col,
        "含义": desc,
        "非空记录数": s.notna().sum(),
        "原始最小日期": s.min(),
        "原始最大日期": s.max(),
        "过滤异常后记录数": clean.notna().sum(),
        "过滤异常后最小日期": clean.min(),
        "过滤异常后最大日期": clean.max(),
        "被过滤记录数": s.notna().sum() - clean.notna().sum(),
    })

date_summary = pd.DataFrame(date_summary_rows)
display(date_summary.style.format({
    "非空记录数": "{:,}",
    "过滤异常后记录数": "{:,}",
    "被过滤记录数": "{:,}",
}))

award_year_counts = (
    pd.to_datetime(ugca["TENDER_DATE_OF_AWARD"], errors="coerce")
    .dt.year
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("授标年份")
    .reset_index(name="合同记录数")
)
display(award_year_counts.style.format({"合同记录数": "{:,}"}))

,字段,含义,非空记录数,原始最小日期,原始最大日期,过滤异常后记录数,过滤异常后最小日期,过滤异常后最大日期,被过滤记录数
0,TENDER_DATE_OF_AWARD,授标日期,"1,242,215",1900-01-01 00:00:00,2030-07-31 00:00:00,"1,242,210",2009-01-15 00:00:00,2025-12-16 00:00:00,5
1,TENDER_DATE_OF_DISPATCH,信息发布日期,"1,272,627",2009-12-31 00:00:00,2025-12-16 00:00:00,"1,272,627",2009-12-31 00:00:00,2025-12-16 00:00:00,0
2,TENDER_MIN_DELIVERY_DATE,数据最早交付日期,"1,272,627",2010-01-01 00:00:00,2025-12-17 00:00:00,"1,272,627",2010-01-01 00:00:00,2025-12-17 00:00:00,0
3,TENDER_CONTRACT_START_DATE,合同开始日期,"1,085,558",1900-01-01 00:00:00,2030-09-01 00:00:00,"1,085,497",2009-01-01 00:00:00,2026-11-01 00:00:00,61
4,TENDER_CONTRACT_END_DATE,合同结束日期,"1,051,044",2007-11-20 00:00:00,9019-09-29 00:00:00,"1,038,998",2009-01-01 00:00:00,2026-12-31 00:00:00,"12,046"


,授标年份,合同记录数
0,1900.000000,1
1,2002.000000,1
2,2004.000000,1
3,2008.000000,1
4,2009.000000,114
5,2010.000000,"68,743"
6,2011.000000,"68,617"
7,2012.000000,"70,200"
8,2013.000000,"67,957"
9,2014.000000,"76,710"


## 采购合同描述中的 AI 相关合同识别

数据字典中与合同文本最相关的字段是 `TENDER_TITLE`、`TENDER_DESCRIPTION`、`TENDER_INDUSTRY_CODES`。本文件 schema 中没有现成的语言字段，因此下面先用合同标题、描述和行业代码文本做启发式语言/文字体系识别，估计不同语言占比；然后用多语言 AI 关键词和正则表达式尽量全面检索 AI 相关合同。

注意：关键词法适合做初筛，可能存在两类误差：一是 `AI` 这种短词带来的误报，二是没有明显 AI 术语但实际属于 AI 项目的漏报。因此代码会保留命中的关键词和样例，方便后续人工抽查。

In [9]:
import re
import pandas as pd
from IPython.display import display

text_cols = [
    "TENDER_BIZPORTAL_ID",
    "TRANSACTION_BIZPORTAL_ID",
    "ORIGIN",
    "TENDER_COUNTRY",
    "TENDER_DATE_OF_AWARD",
    "TENDER_TITLE",
    "TENDER_DESCRIPTION",
    "TENDER_INDUSTRY_CODES",
]

try:
    current_file = files[0]
except NameError:
    from pathlib import Path
    folder = Path(r"D:\.01_projects\Tenderalpha_world_contract\unified-government-contract-awards")
    files = sorted(f for f in folder.glob("*.parquet") if not f.name.startswith("._"))
    current_file = files[0]

try:
    tenders_text = df[text_cols].copy()
except NameError:
    tenders_text = pd.read_parquet(current_file, columns=text_cols)

for col in ["TENDER_TITLE", "TENDER_DESCRIPTION", "TENDER_INDUSTRY_CODES"]:
    tenders_text[col] = tenders_text[col].fillna("").astype(str)

tenders_text["SEARCH_TEXT"] = (
    tenders_text["TENDER_TITLE"] + " \n" +
    tenders_text["TENDER_DESCRIPTION"] + " \n" +
    tenders_text["TENDER_INDUSTRY_CODES"]
)

print(f"当前分析文件: {current_file.name}")
print(f"记录行数: {len(tenders_text):,}")
print("文本字段非空率:")
display(tenders_text[["TENDER_TITLE", "TENDER_DESCRIPTION", "TENDER_INDUSTRY_CODES"]].ne("").mean().rename("非空率").to_frame().style.format("{:.2%}"))

当前分析文件: unified-government-contract-awards_0_0_0.snappy.parquet
记录行数: 1,272,627
文本字段非空率:


,非空率
TENDER_TITLE,100.00%
TENDER_DESCRIPTION,96.17%
TENDER_INDUSTRY_CODES,80.64%


In [10]:
language_field_candidates = [c for c in tenders_text.columns if "LANG" in c.upper()]
print("现成语言字段候选:", language_field_candidates if language_field_candidates else "无")

latin_language_markers = {
    "English/Latin": [" the ", " and ", " for ", " with ", " services", " supply", " support", " maintenance"],
    "Spanish/Latin": [" de ", " para ", " del ", " los ", " las ", " servicio", " suministro", " inteligencia artificial"],
    "Portuguese/Latin": [" de ", " para ", " do ", " dos ", " das ", " servico", " serviço", " fornecimento", " inteligencia artificial", " inteligência artificial"],
    "French/Latin": [" de ", " des ", " pour ", " avec ", " fourniture", " services", " intelligence artificielle"],
    "German/Latin": [" und ", " der ", " die ", " das ", " fuer ", " für ", " leistung", " kuenstliche intelligenz", " künstliche intelligenz"],
    "Romanian/Latin": [" si ", " și ", " pentru ", " contract", " servicii", " furnizare", " inteligenta artificiala", " inteligență artificială"],
    "Polish/Latin": [" oraz ", " dla ", " uslug", " usług", " dostawa", " sztuczna inteligencja"],
}

def detect_text_language(text):
    if not isinstance(text, str) or not text.strip():
        return "Missing text"

    sample = " " + text[:3000].lower() + " "
    cjk = len(re.findall(r"[\u4e00-\u9fff]", sample))
    hangul = len(re.findall(r"[\uac00-\ud7af]", sample))
    cyrillic = len(re.findall(r"[\u0400-\u04ff]", sample))
    greek = len(re.findall(r"[\u0370-\u03ff]", sample))
    arabic = len(re.findall(r"[\u0600-\u06ff]", sample))
    hebrew = len(re.findall(r"[\u0590-\u05ff]", sample))

    script_counts = {
        "CJK Chinese/Japanese": cjk,
        "Korean": hangul,
        "Cyrillic Russian/Ukrainian/etc.": cyrillic,
        "Greek": greek,
        "Arabic/Persian": arabic,
        "Hebrew": hebrew,
    }
    top_script, top_count = max(script_counts.items(), key=lambda x: x[1])
    if top_count >= 3:
        return top_script

    scores = {
        lang: sum(sample.count(marker) for marker in markers)
        for lang, markers in latin_language_markers.items()
    }
    best_lang, best_score = max(scores.items(), key=lambda x: x[1])
    return best_lang if best_score > 0 else "Other/Unknown Latin"

tenders_text["ESTIMATED_LANGUAGE"] = tenders_text["SEARCH_TEXT"].map(detect_text_language)

language_share = (
    tenders_text["ESTIMATED_LANGUAGE"]
    .value_counts(dropna=False)
    .rename_axis("估计语言/文字体系")
    .reset_index(name="合同记录数")
)
language_share["占比"] = language_share["合同记录数"] / len(tenders_text)
display(language_share.style.format({"合同记录数": "{:,}", "占比": "{:.2%}"}))

language_by_country = pd.crosstab(
    tenders_text["TENDER_COUNTRY"].fillna("Missing"),
    tenders_text["ESTIMATED_LANGUAGE"],
    normalize="index",
).round(3)
display(language_by_country.loc[tenders_text["TENDER_COUNTRY"].value_counts().head(15).index])

现成语言字段候选: 无


,估计语言/文字体系,合同记录数,占比
0,English/Latin,"1,076,000",84.55%
1,Other/Unknown Latin,"162,846",12.80%
2,Cyrillic Russian/Ukrainian/etc.,"10,337",0.81%
3,Spanish/Latin,"10,158",0.80%
4,Romanian/Latin,"4,503",0.35%
5,French/Latin,"4,434",0.35%
6,Portuguese/Latin,"1,757",0.14%
7,German/Latin,"1,610",0.13%
8,Polish/Latin,686,0.05%
9,Greek,289,0.02%


ESTIMATED_LANGUAGE,CJK Chinese/Japanese,Cyrillic Russian/Ukrainian/etc.,English/Latin,French/Latin,German/Latin,Greek,Korean,Missing text,Other/Unknown Latin,Polish/Latin,Portuguese/Latin,Romanian/Latin,Spanish/Latin
TENDER_COUNTRY,,,,,,,,,,,,,
US,0.0,0.000,0.915,0.000,0.000,0.0,0.0,0.0,0.083,0.000,0.000,0.002,0.000
KR,0.0,0.000,0.638,0.000,0.000,0.0,0.0,0.0,0.356,0.000,0.000,0.006,0.000
RU,0.0,0.138,0.700,0.000,0.000,0.0,0.0,0.0,0.162,0.000,0.000,0.000,0.000
UA,0.0,0.001,0.786,0.000,0.000,0.0,0.0,0.0,0.212,0.000,0.000,0.000,0.000
CO,0.0,0.000,0.995,0.000,0.000,0.0,0.0,0.0,0.003,0.000,0.000,0.000,0.002
RO,0.0,0.000,0.486,0.055,0.000,0.0,0.0,0.0,0.092,0.000,0.000,0.055,0.311
CL,0.0,0.000,0.422,0.024,0.000,0.0,0.0,0.0,0.034,0.000,0.001,0.000,0.519
IN,0.0,0.000,0.453,0.000,0.000,0.0,0.0,0.0,0.537,0.000,0.003,0.002,0.005
AU,0.0,0.000,0.952,0.000,0.000,0.0,0.0,0.0,0.047,0.000,0.000,0.001,0.000


In [11]:
# 多语言 AI 相关词表。短词 AI 只按独立词匹配，避免匹配到 maintenance、chair 等普通词。
ai_keyword_groups = {
    "core_ai_en": [
        r"\bAI\b", r"\bA\.I\.\b", r"artificial intelligence", r"machine learning", r"deep learning",
        r"generative ai", r"genai", r"large language model", r"\bLLM\b", r"natural language processing", r"\bNLP\b",
        r"computer vision", r"neural network", r"neural networks", r"predictive analytics", r"cognitive computing",
        r"intelligent automation", r"chatbot", r"chat bot", r"virtual assistant", r"robotic process automation", r"\bRPA\b",
        r"data science", r"algorithmic decision", r"automated decision", r"image recognition", r"speech recognition",
        r"text analytics", r"sentiment analysis", r"recommendation engine", r"reinforcement learning",
    ],
    "ai_cjk_korean": [
        r"人工智能", r"机器学习", r"機器學習", r"深度学习", r"深度學習", r"大语言模型", r"大語言模型",
        r"自然语言处理", r"自然語言處理", r"计算机视觉", r"計算機視覺", r"神经网络", r"神經網絡", r"智能算法",
        r"生成式AI", r"生成式人工智能", r"챗봇", r"인공지능", r"머신러닝", r"기계학습", r"딥러닝", r"자연어 처리", r"컴퓨터 비전",
    ],
    "ai_cyrillic": [
        r"искусственн\w* интеллект", r"машинн\w* обуч", r"глубок\w* обуч", r"нейронн\w* сет",
        r"компьютерн\w* зрени", r"обработк\w* естественн\w* язык", r"чат[ -]?бот",
        r"штучн\w* інтелект", r"машинн\w* навчан", r"глибок\w* навчан", r"нейронн\w* мереж",
    ],
    "ai_western_europe": [
        r"inteligencia artificial", r"aprendizaje automatico", r"aprendizaje automático", r"aprendizaje profundo",
        r"procesamiento del lenguaje natural", r"vision artificial", r"visión artificial", r"red neuronal",
        r"inteligência artificial", r"aprendizado de maquina", r"aprendizado de máquina", r"aprendizagem automatica", r"aprendizagem automática",
        r"intelligence artificielle", r"apprentissage automatique", r"apprentissage profond", r"traitement du langage naturel", r"vision par ordinateur",
        r"künstliche intelligenz", r"kuenstliche intelligenz", r"maschinelles lernen", r"neuronale netz", r"sprachverarbeitung",
    ],
    "ai_eastern_europe": [
        r"sztuczna inteligencja", r"uczenie maszynowe", r"sieci neuronowe",
        r"inteligenta artificiala", r"inteligență artificială", r"invatare automata", r"învățare automată", r"retele neuronale", r"rețele neuronale",
        r"umělá inteligence", r"strojové učení", r"neuronové sítě",
    ],
}

compiled_ai_patterns = [
    (group, term, re.compile(term, flags=re.IGNORECASE))
    for group, terms in ai_keyword_groups.items()
    for term in terms
]

def find_ai_terms(text):
    matches = []
    for group, term, pattern in compiled_ai_patterns:
        if pattern.search(text):
            matches.append(f"{group}: {term}")
    return matches

tenders_text["AI_MATCH_TERMS"] = tenders_text["SEARCH_TEXT"].map(find_ai_terms)
tenders_text["IS_AI_RELATED"] = tenders_text["AI_MATCH_TERMS"].map(bool)

ai_total = int(tenders_text["IS_AI_RELATED"].sum())
ai_unique_tenders = tenders_text.loc[tenders_text["IS_AI_RELATED"], "TENDER_BIZPORTAL_ID"].nunique(dropna=True)
print(f"AI 相关合同记录数: {ai_total:,}")
print(f"AI 相关唯一合同/标的 ID 数: {ai_unique_tenders:,}")
print(f"AI 相关记录占比: {ai_total / len(tenders_text):.4%}")

AI 相关合同记录数: 1,994
AI 相关唯一合同/标的 ID 数: 1,960
AI 相关记录占比: 0.1567%


In [12]:
ai_by_language = (
    tenders_text.groupby("ESTIMATED_LANGUAGE", dropna=False)
    .agg(
        合同记录数=("TENDER_BIZPORTAL_ID", "size"),
        AI相关记录数=("IS_AI_RELATED", "sum"),
        AI相关唯一合同数=("TENDER_BIZPORTAL_ID", lambda s: s[tenders_text.loc[s.index, "IS_AI_RELATED"]].nunique(dropna=True)),
    )
    .reset_index()
)
ai_by_language["AI占比"] = ai_by_language["AI相关记录数"] / ai_by_language["合同记录数"]
display(ai_by_language.sort_values("AI相关记录数", ascending=False).style.format({
    "合同记录数": "{:,}",
    "AI相关记录数": "{:,}",
    "AI相关唯一合同数": "{:,}",
    "AI占比": "{:.4%}",
}))

ai_by_country = (
    tenders_text.groupby("TENDER_COUNTRY", dropna=False)
    .agg(
        合同记录数=("TENDER_BIZPORTAL_ID", "size"),
        AI相关记录数=("IS_AI_RELATED", "sum"),
        AI相关唯一合同数=("TENDER_BIZPORTAL_ID", lambda s: s[tenders_text.loc[s.index, "IS_AI_RELATED"]].nunique(dropna=True)),
    )
    .reset_index()
)
ai_by_country["AI占比"] = ai_by_country["AI相关记录数"] / ai_by_country["合同记录数"]
display(ai_by_country.sort_values("AI相关记录数", ascending=False).head(30).style.format({
    "合同记录数": "{:,}",
    "AI相关记录数": "{:,}",
    "AI相关唯一合同数": "{:,}",
    "AI占比": "{:.4%}",
}))

,ESTIMATED_LANGUAGE,合同记录数,AI相关记录数,AI相关唯一合同数,AI占比
2,English/Latin,"1,076,000","1,636","1,616",0.1520%
8,Other/Unknown Latin,"162,846",178,178,0.1093%
12,Spanish/Latin,"10,158",100,93,0.9844%
11,Romanian/Latin,"4,503",46,44,1.0215%
3,French/Latin,"4,434",32,27,0.7217%
4,German/Latin,"1,610",2,2,0.1242%
0,CJK Chinese/Japanese,1,0,0,0.0000%
1,Cyrillic Russian/Ukrainian/etc.,"10,337",0,0,0.0000%
5,Greek,289,0,0,0.0000%
6,Korean,5,0,0,0.0000%


,TENDER_COUNTRY,合同记录数,AI相关记录数,AI相关唯一合同数,AI占比
199,US,"860,297",806,793,0.0937%
163,RU,"74,555",671,671,0.9000%
94,IT,"2,083",154,147,7.3932%
104,KR,"154,483",140,140,0.0906%
161,RO,"10,018",110,98,1.0980%
65,GB,"5,611",29,29,0.5168%
39,CN,"6,231",16,16,0.2568%
40,CO,"14,561",8,8,0.0549%
10,AU,"8,064",6,6,0.0744%
1,AF,"2,748",5,4,0.1820%


In [13]:
ai_by_year = (
    tenders_text.assign(授标年份=pd.to_datetime(tenders_text["TENDER_DATE_OF_AWARD"], errors="coerce").dt.year)
    .groupby("授标年份", dropna=False)
    .agg(
        合同记录数=("TENDER_BIZPORTAL_ID", "size"),
        AI相关记录数=("IS_AI_RELATED", "sum"),
        AI相关唯一合同数=("TENDER_BIZPORTAL_ID", lambda s: s[tenders_text.loc[s.index, "IS_AI_RELATED"]].nunique(dropna=True)),
    )
    .reset_index()
)
ai_by_year["AI占比"] = ai_by_year["AI相关记录数"] / ai_by_year["合同记录数"]
display(ai_by_year.style.format({
    "合同记录数": "{:,}",
    "AI相关记录数": "{:,}",
    "AI相关唯一合同数": "{:,}",
    "AI占比": "{:.4%}",
}))

ai_examples = tenders_text.loc[tenders_text["IS_AI_RELATED"], [
    "TENDER_BIZPORTAL_ID", "TENDER_COUNTRY", "ESTIMATED_LANGUAGE", "TENDER_DATE_OF_AWARD",
    "TENDER_TITLE", "TENDER_DESCRIPTION", "AI_MATCH_TERMS"
]].copy()
ai_examples["AI_MATCH_TERMS"] = ai_examples["AI_MATCH_TERMS"].map(lambda xs: "; ".join(xs[:5]))
display(ai_examples.head(50))

,授标年份,合同记录数,AI相关记录数,AI相关唯一合同数,AI占比
0,1900.000000,1,0,0,0.0000%
1,2002.000000,1,0,0,0.0000%
2,2004.000000,1,0,0,0.0000%
3,2008.000000,1,0,0,0.0000%
4,2009.000000,114,0,0,0.0000%
5,2010.000000,"68,743",49,49,0.0713%
6,2011.000000,"68,617",40,40,0.0583%
7,2012.000000,"70,200",44,44,0.0627%
8,2013.000000,"67,957",61,61,0.0898%
9,2014.000000,"76,710",74,74,0.0965%


,TENDER_BIZPORTAL_ID,TENDER_COUNTRY,ESTIMATED_LANGUAGE,TENDER_DATE_OF_AWARD,TENDER_TITLE,TENDER_DESCRIPTION,AI_MATCH_TERMS
521,191976724,IN,English/Latin,2025-02-11,Re excavation Development of Bagashola Majher ...,,core_ai_en: \bAI\b
622,79725901,US,English/Latin,2021-08-25,"RESEARCH AND DEVELOPMENT IN THE PHYSICAL, ENGI...",FY21 BAA MINE HEALTH AND SAFETY BIG DATA ANALY...,core_ai_en: machine learning
1481,85466382,RO,Romanian/Latin,2022-03-10,Romania-București: Construction materials and ...,Autoritatea contractantă intenționează să înch...,core_ai_en: \bAI\b
3627,85484463,IT,Spanish/Latin,2021-04-27,Italy-Rome: Miscellaneous spare parts,Bando di gara n. 130/2020 _ Procedura aperta a...,core_ai_en: \bAI\b
3927,85484287,IT,English/Latin,2022-03-01,Italy-Rome: Health and safety services,Procedura aperta - in modalità telematica - pe...,core_ai_en: \bAI\b
4431,87467396,US,English/Latin,2022-12-22,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8508930196!WINDSHIELD PANEL,AI",core_ai_en: \bAI\b
4926,47202747,US,English/Latin,2022-12-22,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"DISPLAY,MULTIPLE AI",core_ai_en: \bAI\b
7849,92119146,IT,Other/Unknown Latin,2023-02-02,Italy-Cagliari: Surgical instruments,"Procedura aperta, ai sensi dell'art. 60, D.Lgs...",core_ai_en: \bAI\b
7906,6760196,US,English/Latin,2014-07-29,Aircraft Engine and Engine Parts Manufacturing,"BLADE,COMPRESSOR,AI",core_ai_en: \bAI\b
8080,51115842,US,English/Latin,2019-03-14,AIRCRAFT MANUFACTURING,"4545272714!PLATE,STRUCTURAL,AI",core_ai_en: \bAI\b


## 复核 `ai_examples`：这些合同到底采购什么

上一节的 `ai_examples` 是关键词初筛结果。这里进一步检查命中文本，将合同分为：明确 AI 相关、疑似 AI、明显误报，并给出采购内容类别。尤其要注意两类常见误报：非英语文本中的普通词 `ai`，以及航空/机械零部件描述末尾的 `,AI` 代码。

In [14]:
def _join_text_fields(frame):
    parts = []
    for col in ["TENDER_TITLE", "TENDER_DESCRIPTION", "TENDER_INDUSTRY_CODES"]:
        if col in frame.columns:
            parts.append(frame[col].fillna("").astype(str))
    return parts[0].str.cat(parts[1:], sep=" \n") if len(parts) > 1 else parts[0]

try:
    ai_review = ai_examples.copy()
except NameError:
    ai_review = tenders_text.loc[tenders_text["IS_AI_RELATED"]].copy()

if "SEARCH_TEXT" not in ai_review.columns:
    ai_review["SEARCH_TEXT"] = _join_text_fields(ai_review)

strong_ai_pattern = re.compile(
    r"artificial intelligence|machine learning|deep learning|generative ai|genai|large language model|"
    r"natural language processing|computer vision|neural network|predictive analytics|cognitive computing|"
    r"intelligent automation|chatbot|chat bot|virtual assistant|robotic process automation|data science|"
    r"algorithmic decision|automated decision|image recognition|speech recognition|text analytics|sentiment analysis|"
    r"recommendation engine|reinforcement learning|人工智能|机器学习|深度学习|大语言模型|自然语言处理|"
    r"计算机视觉|神经网络|生成式AI|인공지능|머신러닝|기계학습|딥러닝|컴퓨터 비전|"
    r"искусственн\w* интеллект|машинн\w* обуч|штучн\w* інтелект|inteligencia artificial|"
    r"inteligência artificial|intelligence artificielle|künstliche intelligenz|sztuczna inteligencja|"
    r"inteligenta artificiala|inteligență artificială|umělá inteligence",
    flags=re.IGNORECASE,
)

non_english_ai_word_pattern = re.compile(r"\bai\s+(sensi|fini|sens|faptul|caror|căror|fiecare|fost|centri|siti)\b", flags=re.IGNORECASE)
aircraft_part_ai_pattern = re.compile(r"\b(aircraft|spacecraft|auxiliary equipment|spare parts|panel,ai|plate,?\s*structural,?\s*ai|blade,?\s*compressor,?\s*ai|windshield panel,?\s*ai|support assembly,?\s*ai|nut,?\s*self-locking,?\s*ai)\b", flags=re.IGNORECASE)
rpa_aircraft_pattern = re.compile(r"\b(remotely piloted aircraft|aircraft|squadron|gcs|ground control station|ang|air national guard)\b", flags=re.IGNORECASE)

def review_ai_match(row):
    text = row.get("SEARCH_TEXT", "") or ""
    lower = text.lower()
    has_strong_ai = bool(strong_ai_pattern.search(text))
    has_upper_ai = bool(re.search(r"\bAI\b|\bA\.I\.\b", text))
    has_rpa = bool(re.search(r"\bRPA\b|robotic process automation", text, flags=re.IGNORECASE))

    if non_english_ai_word_pattern.search(text) and not has_strong_ai and not has_upper_ai:
        return "明显误报：非英语普通词 ai"
    if aircraft_part_ai_pattern.search(text) and has_upper_ai and not has_strong_ai:
        return "明显误报：航空/机械零部件代码 AI"
    if has_rpa and rpa_aircraft_pattern.search(text) and "robotic process automation" not in lower:
        return "疑似误报：RPA 可能是 remotely piloted aircraft"
    if has_strong_ai:
        return "明确 AI 相关"
    if has_upper_ai or has_rpa:
        return "疑似 AI：仅缩写命中，需人工复核"
    return "疑似 AI：需人工复核"

def classify_procurement_content(row):
    text = row.get("SEARCH_TEXT", "") or ""
    lower = text.lower()

    if aircraft_part_ai_pattern.search(text):
        return "航空/机械零部件或维修"
    if re.search(r"construction materials|construction|building|engineering studies|energy and related services", lower):
        return "工程/建筑/能源服务"
    if re.search(r"medical|surgical|laboratory|reagents|health|voice recognition|speech recognition", lower):
        return "医疗/实验室/语音识别相关"
    if re.search(r"food|meat|vegetables|fruit|potatoes|poultry|beef|pork", lower):
        return "食品/农产品供应"
    if re.search(r"cleaning|maintenance|parks maintenance|lift-maintenance|security services|postal|courier", lower):
        return "设施维护/清洁/安保/邮政服务"
    if re.search(r"research|development|r&d|baa|prototype|study|studies|experimental development", lower):
        return "AI 研究、研发或原型项目"
    if re.search(r"software|platform|system|systems design|saas|license|licenses|cloud|computer programming|it services|information systems", lower):
        return "AI 软件、平台或信息系统"
    if re.search(r"data analysis|text mining|analytics|data science|predictive|model|models|algorithm", lower):
        return "AI 数据分析、建模或算法服务"
    if re.search(r"robotic process automation|intelligent automation|chatbot|virtual assistant|image recognition|speech recognition|voice recognition", lower):
        return "AI 自动化、识别或虚拟助手应用"
    if re.search(r"consultant|consulting|advisory|strategic partner|training|course|professional services", lower):
        return "AI 咨询、培训或专业服务"
    if re.search(r"gpu|network interface card|server|hardware|chip|uav|drone|5g", lower):
        return "AI 相关硬件/算力/无人系统"
    return "其他/需人工查看描述"

ai_review["AI_REVIEW_LABEL"] = ai_review.apply(review_ai_match, axis=1)
ai_review["PROCUREMENT_CONTENT"] = ai_review.apply(classify_procurement_content, axis=1)

review_summary = (
    ai_review.groupby(["AI_REVIEW_LABEL", "PROCUREMENT_CONTENT"], dropna=False)
    .agg(记录数=("TENDER_BIZPORTAL_ID", "size"), 唯一合同数=("TENDER_BIZPORTAL_ID", "nunique"))
    .reset_index()
    .sort_values(["记录数", "唯一合同数"], ascending=False)
)
display(review_summary.style.format({"记录数": "{:,}", "唯一合同数": "{:,}"}))

,AI_REVIEW_LABEL,PROCUREMENT_CONTENT,记录数,唯一合同数
0,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,568,561
24,疑似 AI：仅缩写命中，需人工复核,AI 研究、研发或原型项目,466,465
26,疑似 AI：仅缩写命中，需人工复核,其他/需人工查看描述,398,398
36,疑似 AI：需人工复核,其他/需人工查看描述,81,73
4,明显误报：非英语普通词 ai,其他/需人工查看描述,68,66
12,明确 AI 相关,AI 研究、研发或原型项目,65,64
14,明确 AI 相关,AI 软件、平台或信息系统,36,34
15,明确 AI 相关,其他/需人工查看描述,26,26
28,疑似 AI：仅缩写命中，需人工复核,工程/建筑/能源服务,26,26
27,疑似 AI：仅缩写命中，需人工复核,医疗/实验室/语音识别相关,26,25


In [15]:
true_or_likely_ai = ai_review[ai_review["AI_REVIEW_LABEL"].isin(["明确 AI 相关", "疑似 AI：仅缩写命中，需人工复核"])]

content_summary = (
    true_or_likely_ai.groupby("PROCUREMENT_CONTENT", dropna=False)
    .agg(记录数=("TENDER_BIZPORTAL_ID", "size"), 唯一合同数=("TENDER_BIZPORTAL_ID", "nunique"))
    .reset_index()
    .sort_values("记录数", ascending=False)
)
display(content_summary.style.format({"记录数": "{:,}", "唯一合同数": "{:,}"}))

cols_to_show = [
    "TENDER_BIZPORTAL_ID", "TENDER_COUNTRY", "TENDER_DATE_OF_AWARD",
    "AI_REVIEW_LABEL", "PROCUREMENT_CONTENT", "TENDER_TITLE", "TENDER_DESCRIPTION",
    "TENDER_INDUSTRY_CODES",
]
existing_cols_to_show = [c for c in cols_to_show if c in ai_review.columns]

display(
    ai_review
    .sort_values(["AI_REVIEW_LABEL", "PROCUREMENT_CONTENT", "TENDER_DATE_OF_AWARD"], ascending=[True, True, False])
    [existing_cols_to_show]
    .head(100)
)

,PROCUREMENT_CONTENT,记录数,唯一合同数
3,AI 研究、研发或原型项目,531,529
6,其他/需人工查看描述,424,424
5,AI 软件、平台或信息系统,59,56
7,医疗/实验室/语音识别相关,43,42
8,工程/建筑/能源服务,36,36
0,AI 咨询、培训或专业服务,16,16
10,设施维护/清洁/安保/邮政服务,14,13
1,AI 数据分析、建模或算法服务,12,12
2,AI 相关硬件/算力/无人系统,6,6
9,航空/机械零部件或维修,6,6


,TENDER_BIZPORTAL_ID,TENDER_COUNTRY,TENDER_DATE_OF_AWARD,AI_REVIEW_LABEL,PROCUREMENT_CONTENT,TENDER_TITLE,TENDER_DESCRIPTION
793003,207006883,US,2025-08-21,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8511580142!PLATE,STRUCTURAL,AI"
781047,205589390,US,2025-07-21,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8511515999!PANEL,STRUCTURAL,AI"
778681,205334606,US,2025-07-14,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,AIRCRAFT MANUFACTURING,"8511490582!BLADE,COMPRESSOR,AI"
775170,202769284,US,2025-07-03,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,AIRCRAFT ENGINE AND ENGINE PARTS MANUFACTURING,"8511379834!BLADE,COMPRESSOR,AI"
771856,204433906,US,2025-06-24,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,AIRCRAFT MANUFACTURING,"8511452303!PLATE,STRUCTURAL,AI"
...,...,...,...,...,...,...,...
679587,81684513,US,2021-04-19,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,AIRCRAFT MANUFACTURING,"4553214181!PLATE,STRUCTURAL,AI"
231186,81134989,US,2021-03-10,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8508002666!WINDSHIELD PANEL,AI"
226593,81101332,US,2021-03-09,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8508043173!BRACKET ASSEMBLY,AI"
909843,74796722,US,2021-03-09,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8507122853!PANEL,STRUCTURAL,AI"


In [17]:
# 单独查看明显误报，便于判断是否需要回到上一节收紧 AI 检索规则。
false_positive_examples = ai_review[ai_review["AI_REVIEW_LABEL"].str.contains("误报", na=False)]
display(false_positive_examples[existing_cols_to_show].head(80))
false_positive_examples.to_csv("明显误报样本.csv", index=False, encoding="utf-8-sig")

# 单独查看明确 AI 合同：这些更适合作为后续统计口径。
clear_ai_examples = ai_review[ai_review["AI_REVIEW_LABEL"].eq("明确 AI 相关")]
display(clear_ai_examples[existing_cols_to_show].head(80))
clear_ai_examples.to_csv("明确AI相关样本.csv", index=False, encoding="utf-8-sig")

,TENDER_BIZPORTAL_ID,TENDER_COUNTRY,TENDER_DATE_OF_AWARD,AI_REVIEW_LABEL,PROCUREMENT_CONTENT,TENDER_TITLE,TENDER_DESCRIPTION
3627,85484463,IT,2021-04-27,明显误报：非英语普通词 ai,航空/机械零部件或维修,Italy-Rome: Miscellaneous spare parts,Bando di gara n. 130/2020 _ Procedura aperta a...
3927,85484287,IT,2022-03-01,明显误报：非英语普通词 ai,医疗/实验室/语音识别相关,Italy-Rome: Health and safety services,Procedura aperta - in modalità telematica - pe...
4431,87467396,US,2022-12-22,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8508930196!WINDSHIELD PANEL,AI"
4926,47202747,US,2022-12-22,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"DISPLAY,MULTIPLE AI"
7849,92119146,IT,2023-02-02,明显误报：非英语普通词 ai,医疗/实验室/语音识别相关,Italy-Cagliari: Surgical instruments,"Procedura aperta, ai sensi dell'art. 60, D.Lgs..."
...,...,...,...,...,...,...,...
115844,7478240,US,2014-11-19,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,Other Aircraft Part and Auxiliary Equipment Ma...,"8500770588!PANEL,STRUCTURAL,AI"
117009,20106027,GB,2014-11-18,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,Other Aircraft Part and Auxiliary Equipment Ma...,"8501560556!PLATE,STRUCTURAL,AI"
118882,118260081,US,2023-11-30,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,OTHER AIRCRAFT PARTS AND AUXILIARY EQUIPMENT M...,"8510297724!COVER,SEAT FRAME,AI"
121999,13188039,US,2017-06-23,明显误报：航空/机械零部件代码 AI,航空/机械零部件或维修,Other Aircraft Part and Auxiliary Equipment Ma...,"8504517973!BEARING,PITCH,DF,AI"


,TENDER_BIZPORTAL_ID,TENDER_COUNTRY,TENDER_DATE_OF_AWARD,AI_REVIEW_LABEL,PROCUREMENT_CONTENT,TENDER_TITLE,TENDER_DESCRIPTION
622,79725901,US,2021-08-25,明确 AI 相关,医疗/实验室/语音识别相关,"RESEARCH AND DEVELOPMENT IN THE PHYSICAL, ENGI...",FY21 BAA MINE HEALTH AND SAFETY BIG DATA ANALY...
15388,192064269,NaN,2024-06-19,明确 AI 相关,AI 研究、研发或原型项目,Consultant for the Center for Artificial Intel...,
18571,87515131,US,2023-03-29,明确 AI 相关,AI 软件、平台或信息系统,COMPUTER SYSTEMS DESIGN SERVICES,NEXT GENERATION NETWORK SERVICES FOR THREE PRO...
20446,85573623,GB,2022-03-30,明确 AI 相关,医疗/实验室/语音识别相关,WinvoiceWeb - Dragon Medical One Voice Recogni...,North Tees & Hartlepool NHS Foundation Trust h...
30449,85640837,GB,2022-03-21,明确 AI 相关,AI 数据分析、建模或算法服务,EMRAD Training Courses,Training programmes delivered via a mixture of...
...,...,...,...,...,...,...,...
534064,84468169,US,2022-04-20,明确 AI 相关,AI 软件、平台或信息系统,ENSURING CONSISTENCY OF SYSTEMIC INFORMATION T...,ENSURING CONSISTENCY OF SYSTEMIC INFORMATION T...
548427,88232018,AU,2022-07-24,明确 AI 相关,AI 数据分析、建模或算法服务,Computer services : Department of Defence : 20...,Data Science Support Services
556774,47260043,US,2018-08-29,明确 AI 相关,AI 软件、平台或信息系统,SOFTWARE PUBLISHERS,"TWO (2) MATLAB (MLSMS), TWO (2) IMAGE PROCESSI..."
575712,162900883,KR,2011-11-15,明确 AI 相关,其他/需人工查看描述,"Seoul Artificial Intelligence High School, Seo...",Purchase of 14 computers and 2 other types of...
